# Task11 - [Advanced] Driver Consistency Score

This consistency score combines two factors — the variability of a driver's finishing positions and their reliability in finishing races — weighted 60% and 40% respectively.

**Standard deviation score** (lower variance → higher score):

$$
\text{std\_score} = \frac{1}{1 + \sigma_{\text{position}}}
$$

**DNF (reliability) score:**

$$
\text{dnf\_score} = 1 - \frac{\text{DNF count}}{\text{Total races}}
$$

**Final consistency score:**

$$
\text{Consistency Score} = 0.6 \times \text{std\_score} + 0.4 \times \text{dnf\_score}
$$

Because raw position variance alone made backmarkers who consistently finished near the back look more "consistent" than actual front-runners, the analysis was restricted to each season's **top 10 point scorers**, so the score measures stability among drivers who were already competitive rather than just steady mediocrity. Validating the results against real F1 history (Schumacher's 2002 season, Vettel's 2011 and 2013 seasons, Verstappen's 2023 season) showed the top scores aligned closely with widely recognized dominant seasons, confirming the metric behaves as intended.

In [1]:
# Design your own formula/metric to measure a driver's "consistency" across a season (you decide what factors matter - e.g. variance in finishing position, DNF rate, points per race, etc)
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_f1.csv')

subset = df[['year', 'full_name', 'raceId', 'position', 'statusId', 'points']]

df_status = pd.read_csv('../data/status.csv')
subset = subset.merge(df_status, on='statusId', how='left')

finished_pattern = subset['status'].str.contains(r'^\+\d+ Lap', regex=True)
finished_exact = subset['status'] == 'Finished'
subset['is_finished'] = finished_pattern | finished_exact

result = subset.groupby(['year', 'full_name']).agg(
    avg_position=('position', 'mean'),
    std_position=('position', 'std'),
    race_count=('raceId', 'count'),
    dnf_count=('is_finished', lambda x: (~x).sum()),
    total_points=('points', 'sum')
).reset_index()

result['dnf_rate'] = result['dnf_count'] / result['race_count']
result = result[result['race_count'] >= 5]
result = result[result['total_points'] > 0]

result = result.sort_values(['year', 'total_points'], ascending=[True, False])
result = result.groupby('year').head(10)

result['std_score'] = 1 / (1 + result['std_position'])
result['dnf_score'] = 1 - result['dnf_rate']
result['consistency_score'] = (
    result['std_score'] * 0.6 +
    result['dnf_score'] * 0.4
)

result.to_csv('../data/consistency_scores.csv', index=False)

for year in range(2000, 2025):
    year_data = result[result['year'] == year]
    top5 = year_data.sort_values('consistency_score', ascending=False).head(5)
    print(f"=== {year}년 Top 5 ===")
    print(top5[['full_name', 'consistency_score', 'total_points']])
    print()

=== 2000년 Top 5 ===
                 full_name  consistency_score  total_points
2635    Rubens Barrichello           0.607830          62.0
2627    Michael Schumacher           0.579489         108.0
2628         Mika Häkkinen           0.575582          89.0
2614       David Coulthard           0.555189          73.0
2617  Giancarlo Fisichella           0.408036          18.0

=== 2001년 Top 5 ===
               full_name  consistency_score  total_points
2652  Michael Schumacher           0.680129         123.0
2637     David Coulthard           0.568922          65.0
2659  Rubens Barrichello           0.442723          56.0
2654       Nick Heidfeld           0.441068          12.0
2644  Jacques Villeneuve           0.431105          12.0

=== 2002년 Top 5 ===
               full_name  consistency_score  total_points
2677  Michael Schumacher           0.770749         144.0
2683  Rubens Barrichello           0.502675          77.0
2674  Juan Pablo Montoya           0.483090          50.

# Task12 - [Advanced] Constructor Championship Simulation

In [2]:
# Using historical points-per-position rules, write a function that recalculates the final constructor standings for a given year if the points system were different (e.g. apply 2020 rules to 1990 season, or design your own points scale)
# Compare original standings vs simulated standings for at least 1 season
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_f1.csv')

f1_points_systems = {
    # 1950-1957: top 5 finishers, fastest lap bonus point existed separately
    (1950, 1957): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2},

    # 1958-1959: top 5 finishers, shared-points rule removed
    (1958, 1959): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2},

    # 1960: top 6 finishers, fastest lap bonus removed
    (1960, 1960): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 1961-1990: top 6 finishers, win worth 9 points (longest-running system)
    (1961, 1990): {1: 9, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 1991-2002: top 6 finishers, win worth 10 points
    (1991, 2002): {1: 10, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 2003-2009: expanded to top 8 finishers
    (2003, 2009): {1: 10, 2: 8, 3: 6, 4: 5, 5: 4, 6: 3, 7: 2, 8: 1},

    # 2010-2024: top 10 finishers, win worth 25 points (current era)
    (2010, 2024): {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1},
}

def get_points_system(year):
    # Look up which points system applies to a given year
    for (start, end), system in f1_points_systems.items():
        if start <= year <= end:
            return system
    raise ValueError(f"No points system found for year {year}")

def calculate_points(position, points_system):
    if pd.isna(position):
        return 0
    position = int(position)
    return points_system.get(position, 0)

def simulate_standings(year, points_system):
    # Filter data for the given year and copy to avoid warnings
    season_data = df[df['year'] == year].copy()
    
    # Apply the points system to each row's finishing position
    season_data['simulated_points'] = season_data['position'].apply(
        lambda x: calculate_points(x, points_system)
    )
    
    # Sum up the ORIGINAL points per constructor (real historical result)
    original_standings = season_data.groupby('constructorId')['points'].sum().reset_index()
    original_standings = original_standings.rename(columns={'points': 'original_points'})
    
    # Sum up the SIMULATED points per constructor (using the new system)
    simulated_standings = season_data.groupby('constructorId')['simulated_points'].sum().reset_index()
    
    # Merge both standings side by side on constructor name
    comparison = original_standings.merge(simulated_standings, on='constructorId', how='left')
    
    # Rank both columns (higher points = better rank, i.e. rank 1)
    comparison['original_rank'] = comparison['original_points'].rank(ascending=False, method='min').astype(int)
    comparison['simulated_rank'] = comparison['simulated_points'].rank(ascending=False, method='min').astype(int)
    
    # Sort by original rank for readability
    comparison = comparison.sort_values('original_rank')
    
    return comparison

# Ask the user which year they want to recalculate
year = int(input("Enter the year to recalculate: "))

# Ask which year's points system to apply
system_year = int(input("Which year's points system do you want to apply? (e.g. 2020): "))
points_system = get_points_system(system_year)   

result = simulate_standings(year, points_system)
print(result)

    constructorId  original_points  simulated_points  original_rank  \
0               1            121.0               388              1   
2               6            110.0               343              2   
5              22             71.0               278              3   
1               3             57.0               235              4   
6              25             16.0               100              5   
9              33             11.0                77              6   
13             41              7.0                47              7   
8              32              3.0                43              8   
10             34              2.0                18              9   
4              21              2.0                34              9   
3              18              0.0                14             11   
7              27              0.0                22             11   
17             46              0.0                 6             11   
11    